# Proyecto 8 — Corte 3 Final: Modelo neuronal + LLM + Aprendizaje por Refuerzo

**Integrantes:** Kevin Tapias, Sebastian Villera y Anuar Marriaga

```
 Formulario web ─► Red neuronal (Corte 2) ─► predicción ─┐
 (index.html)       Rápida / Intermedia / Lenta,          │
                    minutos y prob. de entrega lenta      ▼
                                       Agente Q-learning escoge la estrategia (acción)
                                                          │
                                                          ▼
                              Prompt (rol + datos + predicción + estrategia) ─► LLM Qwen2.5 ─► Recomendación
                                                                                                   │
                         Tabla Q se actualiza ◄── recompensa / castigo ◄── usuario califica (★ / 👍 👎)
```

1. **Modelo neuronal (Corte 2):** el ensamble de 6 redes + regresor de minutos predice si el pedido llegará **Rápido (≤70 min)**, **Intermedio** o **Lento (≥180 min)**, con los minutos estimados y la probabilidad de entrega lenta.
2. **LLM pre-entrenado (Qwen2.5-3B-Instruct, Hugging Face)** conectado por **pipeline** de `transformers`: la predicción entra como **input** del prompt y el LLM redacta recomendaciones para manejar el pedido.
3. **Aprendizaje por refuerzo (Q-learning):** el usuario **califica** la recomendación; la calificación es la **recompensa o castigo** que actualiza la **tabla Q** (estados × acciones). Con ε-greedy el agente pasa de **exploración** a **explotación**.
4. **Despliegue web:** API en **FastAPI** (`/predict`, `/feedback`, `/q_tabla`) + página `index.html` con formulario y calificación.

> Abrir en **Google Colab** → Entorno de ejecución → Cambiar tipo → **GPU T4**. Correr celda por celda.

## Paso 1: instalar y cargar librerías

- `transformers` + `accelerate`: descargar y correr el LLM pre-entrenado.
- `fastapi` + `uvicorn`: la API web.
- `scikit-learn==1.6.1`: la misma versión con la que se guardó `preprocesador.joblib` en el Corte 2.

In [ ]:
!pip install -q "scikit-learn==1.6.1" transformers accelerate fastapi uvicorn

In [ ]:
import os, sys, json, time, zipfile, shutil, importlib, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch

random.seed(42); np.random.seed(42); tf.keras.utils.set_random_seed(42); torch.manual_seed(42)
print("TensorFlow:", tf.__version__, "| PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (activar T4)")

## Paso 2: subir el modelo del Corte 2 y el dataset

Subir **dos archivos**:
- `despliegue.zip` (o `despliegue (1).zip`): `clasificador.keras`, `regresor_minutos.keras`, `preprocesador.joblib`, `features.py`, `inferencia.py` y `config.json` del Corte 2.
- `amazon_delivery.csv`: para tomar pedidos reales y probar el sistema.

Todo queda en la carpeta `despliegue_llm_rl/`, que es la aplicación final.

In [ ]:
from google.colab import files

CARPETA = "despliegue_llm_rl"
os.makedirs(f"{CARPETA}/static", exist_ok=True)

import glob
# Se pueden subir los dos a la vez (Ctrl + clic) o de a uno corriendo esta celda dos veces:
# los archivos se buscan en el disco, no solo en lo último que se subió.
faltan = lambda: not glob.glob("*.csv") or not [z for z in glob.glob("*.zip") if z != "despliegue_llm_rl.zip"]
if faltan():
    files.upload()
if faltan():
    raise SystemExit("Falta subir el .zip del modelo o el .csv: vuelve a correr esta celda y súbelo.")

CSV_PATH = sorted(glob.glob("*.csv"))[0]
for nombre in glob.glob("*.zip"):
    if nombre != "despliegue_llm_rl.zip":
        with zipfile.ZipFile(nombre) as z:
            z.extractall("_modelo_corte2")

for raiz, _, archivos in os.walk("_modelo_corte2"):
    for a in archivos:
        if a.endswith((".keras", ".joblib", ".json", ".py")):
            shutil.copy(os.path.join(raiz, a), CARPETA)
print("Archivos del modelo:", sorted(os.listdir(CARPETA)))
print("CSV:", CSV_PATH)
if CARPETA not in sys.path:
    sys.path.insert(0, CARPETA)

## Paso 3: modelo neuronal — predicción con `model.predict`

`Predictor.predecir` construye las variables con `features.py` (las mismas del entrenamiento), aplica el preprocesador y llama a `model.predict` del clasificador y del regresor. Probamos tres pedidos escritos a mano.

In [ ]:
import features, inferencia
importlib.reload(features); importlib.reload(inferencia)
from features import RAW_COLS

predictor = inferencia.Predictor(CARPETA)
CFG = predictor.cfg
print(f"Modelo: {CFG['modelo']} | acierto test: {CFG['acierto_test']:.4f} | error test: {CFG['error_test']:.4f}")

pedidos_demo = pd.DataFrame([
    {"Agent_Age": 25, "Agent_Rating": 4.9, "Store_Latitude": 12.914, "Store_Longitude": 77.678,
     "Drop_Latitude": 12.934, "Drop_Longitude": 77.688, "Order_Date": "2022-03-15",
     "Order_Time": "10:00", "Pickup_Time": "10:10", "Weather": "Sunny", "Traffic": "Low",
     "Vehicle": "motorcycle", "Area": "Urban", "Category": "Grocery"},
    {"Agent_Age": 30, "Agent_Rating": 4.6, "Store_Latitude": 12.914, "Store_Longitude": 77.678,
     "Drop_Latitude": 12.954, "Drop_Longitude": 77.718, "Order_Date": "2022-03-19",
     "Order_Time": "18:30", "Pickup_Time": "18:45", "Weather": "Fog", "Traffic": "Jam",
     "Vehicle": "motorcycle", "Area": "Metropolitian", "Category": "Clothing"},
    {"Agent_Age": 38, "Agent_Rating": 4.1, "Store_Latitude": 22.745, "Store_Longitude": 75.892,
     "Drop_Latitude": 22.875, "Drop_Longitude": 76.022, "Order_Date": "2022-03-20",
     "Order_Time": "19:30", "Pickup_Time": "19:45", "Weather": "Stormy", "Traffic": "Jam",
     "Vehicle": "scooter", "Area": "Metropolitian", "Category": "Electronics"},
])
pred_demo = predictor.predecir(pedidos_demo)
display(pd.concat([pedidos_demo[["Weather", "Traffic", "Vehicle", "Area", "Order_Time"]], pred_demo], axis=1).round(3))

## Paso 4: aprendizaje por refuerzo — módulo Q-learning

Lo escribimos antes que el LLM porque el prompt usa la **acción** que escoge el agente. Se explica en detalle en el Paso 7.

In [ ]:
%%writefile despliegue_llm_rl/rl_feedback.py
"""Aprendizaje por refuerzo (Q-learning) para calificar las recomendaciones del LLM.

- Estados (s): la categoría que predijo la red neuronal -> Rápida, Intermedia, Lenta.
- Acciones (a): estrategias de recomendación predeterminadas (ACCIONES). El agente escoge una
  y el LLM la desarrolla en un texto adaptado al pedido.
- Recompensa (r): el feedback del usuario. Estrellas 1..5 -> r = (estrellas - 3) / 2
  (1★ = -1 castigo fuerte, 3★ = 0 neutro, 5★ = +1 premio). Me gusta = +1, No me gusta = -1.
- Tabla Q: Q[s][a] = qué tan buena es la acción a en el estado s, según lo aprendido.
- Actualización (Q-learning):  Q(s,a) <- Q(s,a) + alfa * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]
  Cada pedido es un episodio de un solo paso (después de recomendar no hay estado siguiente),
  así que s' es terminal y el objetivo queda r. gamma se deja como hiperparámetro por si se
  encadenan pasos (p. ej. seguimiento del mismo pedido).
- Política epsilon-greedy: con probabilidad epsilon explora (acción al azar) y si no, explota
  (acción con mayor Q). epsilon baja con cada feedback: exploración -> explotación.
"""
import json
import os
from datetime import datetime

import numpy as np

ESTADOS = ["Rápida", "Intermedia", "Lenta"]

ACCIONES = {
    "despacho_estandar": "Mantener el despacho estándar y confirmar al cliente la hora estimada de llegada",
    "priorizar_tienda": "Priorizar la preparación y el despacho inmediato del pedido en la tienda",
    "ruta_alternativa": "Planear una ruta alternativa que evite las vías congestionadas y avisar al repartidor",
    "reasignar_repartidor": "Reasignar el pedido a un repartidor o vehículo más adecuado para las condiciones",
    "avisar_cliente": "Avisar al cliente de forma proactiva con una nueva hora estimada y seguimiento en vivo",
    "compensar_reprogramar": "Ofrecer una compensación (cupón o envío gratis) o reprogramar la franja de entrega",
}

# Valores iniciales predeterminados (conocimiento previo del negocio, en la escala de la recompensa).
# Son pequeños para que el feedback del usuario los cambie rápido.
Q_INICIAL = {
    "Rápida":     {"despacho_estandar": 0.30, "priorizar_tienda": 0.10, "ruta_alternativa": 0.00,
                   "reasignar_repartidor": -0.10, "avisar_cliente": 0.10, "compensar_reprogramar": -0.20},
    "Intermedia": {"despacho_estandar": 0.05, "priorizar_tienda": 0.15, "ruta_alternativa": 0.15,
                   "reasignar_repartidor": 0.05, "avisar_cliente": 0.20, "compensar_reprogramar": -0.05},
    "Lenta":      {"despacho_estandar": -0.20, "priorizar_tienda": 0.10, "ruta_alternativa": 0.15,
                   "reasignar_repartidor": 0.10, "avisar_cliente": 0.25, "compensar_reprogramar": 0.20},
}


def estrellas_a_recompensa(estrellas):
    return (float(estrellas) - 3.0) / 2.0


class AgenteQ:
    def __init__(self, ruta="q_tabla.json", alfa=0.3, gamma=0.9,
                 epsilon_inicial=0.5, epsilon_min=0.05, decaimiento=0.95, semilla=None):
        self.ruta = ruta
        self.alfa, self.gamma = alfa, gamma
        self.epsilon_inicial, self.epsilon_min, self.decaimiento = epsilon_inicial, epsilon_min, decaimiento
        self.rng = np.random.default_rng(semilla)
        self.Q = {s: dict(Q_INICIAL[s]) for s in ESTADOS}
        self.N = {s: {a: 0 for a in ACCIONES} for s in ESTADOS}
        self.historial = []
        if ruta and os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as f:
                datos = json.load(f)
            self.Q, self.N, self.historial = datos["Q"], datos["N"], datos["historial"]

    # ---------------- política ----------------
    @property
    def epsilon(self):
        return max(self.epsilon_min, self.epsilon_inicial * self.decaimiento ** len(self.historial))

    def ranking(self, estado):
        """Acciones ordenadas de mayor a menor Q (lo que el agente cree mejor)."""
        return sorted(self.Q[estado], key=self.Q[estado].get, reverse=True)

    def elegir_accion(self, estado):
        """epsilon-greedy. Devuelve (acción, 'exploración' | 'explotación')."""
        if self.rng.random() < self.epsilon:
            return str(self.rng.choice(list(ACCIONES))), "exploración"
        return self.ranking(estado)[0], "explotación"

    # ---------------- aprendizaje ----------------
    def actualizar(self, estado, accion, recompensa, estado_siguiente=None, info=None):
        q_actual = self.Q[estado][accion]
        futuro = 0.0 if estado_siguiente is None else max(self.Q[estado_siguiente].values())
        objetivo = recompensa + self.gamma * futuro
        self.Q[estado][accion] = q_actual + self.alfa * (objetivo - q_actual)
        self.N[estado][accion] += 1
        self.historial.append({"fecha": datetime.now().isoformat(timespec="seconds"),
                               "estado": estado, "accion": accion, "recompensa": float(recompensa),
                               "q_antes": q_actual, "q_despues": self.Q[estado][accion],
                               "epsilon": self.epsilon, **(info or {})})
        self.guardar()
        return self.Q[estado][accion]

    def feedback_estrellas(self, estado, accion, estrellas, comentario=""):
        if not 1 <= int(estrellas) <= 5:
            raise ValueError("Las estrellas deben estar entre 1 y 5")
        r = estrellas_a_recompensa(estrellas)
        return r, self.actualizar(estado, accion, r, info={"tipo": "estrellas", "estrellas": int(estrellas),
                                                           "comentario": comentario})

    def feedback_me_gusta(self, estado, accion, me_gusta):
        r = 1.0 if me_gusta else -1.0
        return r, self.actualizar(estado, accion, r, info={"tipo": "me_gusta" if me_gusta else "no_me_gusta"})

    # ---------------- persistencia ----------------
    def guardar(self):
        if not self.ruta:
            return
        with open(self.ruta, "w", encoding="utf-8") as f:
            json.dump({"Q": self.Q, "N": self.N, "historial": self.historial}, f, ensure_ascii=False, indent=2)

    def tabla(self):
        """Tabla Q como lista de filas (para mostrar en pandas o en la web)."""
        return [{"estado": s, "accion": a, "Q": round(self.Q[s][a], 4), "veces": self.N[s][a]}
                for s in ESTADOS for a in ACCIONES]

## Paso 5: LLM pre-entrenado — Qwen2.5-3B-Instruct

**Descripción del LLM:** usamos **Qwen2.5-3B-Instruct**, un modelo de lenguaje grande (LLM) de código abierto de Alibaba Cloud, publicado en Hugging Face. Es un Transformer tipo *decoder-only* de unos 3 mil millones de parámetros, pre-entrenado con alrededor de 18 billones de tokens en más de 29 idiomas (incluido el español) y ajustado con instrucciones y RLHF para seguir órdenes como un chat. Acepta mensajes con **rol** (`system`, `user`, `assistant`), lo que nos deja darle el papel de experto en logística. Corre en la GPU T4 gratuita de Colab en float16 (~6 GB). **No lo re-entrenamos**: lo conectamos por **pipeline** de `transformers` (`pipeline("text-generation")`), y la predicción de nuestra red neuronal entra como input del prompt. Lo escogimos por ser gratuito, sin API key, liviano y bueno en español; es el mismo tipo de modelo (Qwen) que se mostró en clase. Si se quiere, `MODELO_LLM` se cambia por `Qwen/Qwen2.5-1.5B-Instruct` (más rápido) o `meta-llama/Llama-3.2-3B-Instruct` (pide token de Hugging Face).

**Parámetros de generación:** `max_new_tokens=350`, `temperature=0.4` (poca creatividad, para que no invente) y `top_p=0.9`.

In [ ]:
%%writefile despliegue_llm_rl/recomendador.py
"""LLM pre-entrenado (Hugging Face, por pipeline de transformers) que convierte la predicción
de la red neuronal en un texto con recomendaciones para manejar el pedido."""
import os

import pandas as pd

from features import construir_features
from rl_feedback import ACCIONES

MODELO_LLM = os.environ.get("MODELO_LLM", "Qwen/Qwen2.5-3B-Instruct")

TRADUCCION = {
    "Sunny": "soleado", "Cloudy": "nublado", "Fog": "niebla", "Sandstorms": "tormenta de arena",
    "Stormy": "tormenta", "Windy": "ventoso", "Low": "bajo", "Medium": "medio", "High": "alto",
    "Jam": "atasco", "motorcycle": "moto", "scooter": "scooter", "van": "furgoneta",
    "Metropolitian": "metropolitana", "Urban": "urbana", "Semi-Urban": "semi-urbana", "Other": "otra",
    "Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles", "Thursday": "jueves",
    "Friday": "viernes", "Saturday": "sábado", "Sunday": "domingo",
}

ROL_SISTEMA = (
    "Eres un experto en logística de última milla de una empresa de comercio electrónico en India. "
    "Un modelo de redes neuronales ya predijo cómo llegará un pedido y tú debes escribir recomendaciones "
    "prácticas para manejarlo. Reglas: responde SIEMPRE en español; NO contradigas la predicción; "
    "empieza mencionando el TIPO DE ENTREGA predicho (Rápida, Intermedia o Lenta) y los minutos estimados; "
    "el producto (Clothing, Electronics, Grocery...) es solo lo que se envía, NUNCA es el tipo de entrega; "
    "es una entrega a domicilio: el cliente no recoge el pedido; no inventes datos que no estén "
    "en el pedido; da acciones concretas para la tienda, el repartidor y servicio al cliente; "
    "si la entrega es Rápida no hables de demoras ni ofrezcas descuentos o compensaciones; "
    "no escribas firmas ni plantillas con corchetes; máximo 180 palabras."
)


def cargar_llm(modelo=MODELO_LLM):
    import torch
    from transformers import pipeline
    gpu = torch.cuda.is_available()
    return pipeline("text-generation", model=modelo,
                    torch_dtype=torch.float16 if gpu else torch.float32,
                    device_map="auto" if gpu else None)


def estado_de(resultado):
    """Texto de la red -> estado del agente RL (Rápida / Intermedia / Lenta)."""
    r = str(resultado)
    for estado in ["Rápida", "Intermedia", "Lenta"]:
        if r.startswith(estado):
            return estado
    return None


def _t(v):
    v = str(v).strip()
    return TRADUCCION.get(v, v)


def describir_prediccion(pred):
    estado = estado_de(pred["resultado"])
    lineas = [f"- Tipo de entrega predicho: {pred['resultado']}",
              f"- Tiempo estimado: unos {pred['minutos_estimados']:.0f} minutos (aproximado, ± 18 min)"]
    if estado == "Intermedia":
        lado = "lenta" if pred["prob_lenta"] >= 0.5 else "rápida"
        lineas.append(f"- Ni claramente rápida ni claramente lenta; se inclina hacia {lado} "
                      f"(indicador {pred['prob_lenta']:.0%} hacia lenta)")
    else:
        lineas.append(f"- Probabilidad de entrega lenta (180 min o más): {pred['prob_lenta']:.0%}")
    if estado == "Rápida":
        lineas.append("- Usa el tipo de entrega Rápida como dato principal; los minutos son solo una referencia")
    return "\n".join(lineas)


def describir_pedido(fila_raw, pred):
    f = construir_features(pd.DataFrame([fila_raw])).iloc[0]
    return (
        "DATOS DEL PEDIDO:\n"
        f"- Producto enviado (no es el tipo de entrega): {fila_raw['Category']}\n"
        f"- Distancia tienda-cliente: {f['distance_km']:.1f} km, zona {_t(fila_raw['Area'])}\n"
        f"- Pedido a las {str(fila_raw['Order_Time'])[:5]} del {_t(f['Dia_semana'])}, recogida a las "
        f"{str(fila_raw['Pickup_Time'])[:5]} ({f['espera_pickup_min']:.0f} min de espera)\n"
        f"- Clima: {_t(fila_raw['Weather'])} | Tráfico: {_t(fila_raw['Traffic'])} | "
        f"Vehículo: {_t(fila_raw['Vehicle'])}\n"
        f"- Repartidor: {float(fila_raw['Agent_Age']):.0f} años, calificación {float(fila_raw['Agent_Rating']):.1f}/5\n\n"
        "PREDICCIÓN DEL MODELO NEURONAL:\n" + describir_prediccion(pred)
    )


def construir_prompt(fila_raw, pred, accion):
    usuario = (
        describir_pedido(fila_raw, pred) +
        f"\n\nESTRATEGIA PRINCIPAL (escogida por el agente de aprendizaje por refuerzo): {ACCIONES[accion]}.\n"
        "Escribe la recomendación así:\n"
        "1. Una frase con el diagnóstico (tipo de entrega, minutos y la causa más probable según los datos).\n"
        "2. Tres o cuatro acciones concretas en viñetas, empezando por la estrategia principal.\n"
        "3. Un mensaje corto y amable para el cliente.\n"
        "No agregues más puntos después del 3."
    )
    return [{"role": "system", "content": ROL_SISTEMA}, {"role": "user", "content": usuario}]


def es_coherente(texto, pred, producto=""):
    """La respuesta debe nombrar el tipo de entrega predicho y los minutos estimados, y no confundir
    el producto con el tipo de entrega (p. ej. "La categoría de entrega es Clothing")."""
    raiz = {"Rápida": "rápid", "Intermedia": "intermedi", "Lenta": "lent"}.get(estado_de(pred["resultado"]), "")
    t = texto.lower().replace("rapid", "rápid")
    primera_frase = t.split("\n")[0].split(".")[0]
    producto = str(producto).strip().lower()
    confunde = bool(producto) and producto in primera_frase and (
        "categoría" in primera_frase or "tipo de entrega" in primera_frase)
    return raiz in t and f"{pred['minutos_estimados']:.0f}" in texto and not confunde


def generar_descripcion_llm(llm, fila_raw, pred, accion, max_tokens=350, temperatura=0.4, intentos=3):
    mensajes = construir_prompt(fila_raw, pred, accion)
    texto = ""
    for _ in range(intentos):
        salida = llm(mensajes, max_new_tokens=max_tokens, do_sample=True,
                     temperature=temperatura, top_p=0.9)
        texto = salida[0]["generated_text"][-1]["content"].strip()
        if es_coherente(texto, pred, fila_raw["Category"]):
            break
    return texto

In [ ]:
import rl_feedback, recomendador
importlib.reload(rl_feedback); importlib.reload(recomendador)
from rl_feedback import AgenteQ, ACCIONES, ESTADOS, Q_INICIAL
from recomendador import (MODELO_LLM, cargar_llm, construir_prompt, generar_descripcion_llm,
                          es_coherente, estado_de)

t0 = time.time()
llm = cargar_llm(MODELO_LLM)
print(f"LLM cargado: {MODELO_LLM} | {llm.model.num_parameters()/1e9:.2f} mil millones de parámetros | {time.time()-t0:.0f} s")

## Paso 6: diseño del prompt — la predicción entra como INPUT al LLM

El mensaje tiene:
1. **Rol del sistema:** *"Eres un experto en logística de última milla…"* con reglas: español, no contradecir la predicción, empezar con el tipo de entrega y los minutos, no confundir el producto con el tipo de entrega, no ofrecer compensaciones si la entrega es Rápida, no inventar datos.
2. **Datos del pedido** (producto, distancia, hora, clima, tráfico, vehículo, repartidor) **+ predicción de la red** (tipo de entrega, minutos y probabilidad).
3. **Estrategia principal**: la acción que escogió el agente de Q-learning.
4. **Formato:** diagnóstico, 3–4 acciones concretas y un mensaje para el cliente.

Además, `es_coherente` revisa que el texto nombre el tipo de entrega predicho y los minutos, y que no diga que el producto es la "categoría"; si no, se genera otra vez (hasta 3 intentos).

Así se ve el prompt real del pedido lento:

In [ ]:
mensajes = construir_prompt(pedidos_demo.iloc[2], pred_demo.iloc[2], "avisar_cliente")
for m in mensajes:
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

In [ ]:
for i in range(len(pedidos_demo)):
    fila, pred = pedidos_demo.iloc[i], pred_demo.iloc[i]
    estado = estado_de(pred["resultado"])
    accion = max(Q_INICIAL[estado], key=Q_INICIAL[estado].get)   # mejor acción según la tabla Q inicial
    t0 = time.time()
    texto = generar_descripcion_llm(llm, fila, pred, accion)
    print("=" * 100)
    print(f"PEDIDO {i+1}: {pred['resultado']} | {pred['minutos_estimados']:.0f} min | P(lenta) {pred['prob_lenta']:.0%} "
          f"| acción: {accion} | coherente: {es_coherente(texto, pred, fila['Category'])} | {time.time()-t0:.1f} s")
    print("-" * 100)
    print(texto)

## Paso 7: Aprendizaje por Refuerzo (Q-learning) con el feedback del usuario

No es un modelo supervisado: es un **agente** con una **tabla Q** de valores discretos.

| Elemento | En nuestro proyecto |
|---|---|
| **Estados (s)** | Lo que predijo la red: `Rápida`, `Intermedia`, `Lenta` |
| **Acciones (a)** | 6 estrategias predeterminadas: `despacho_estandar`, `priorizar_tienda`, `ruta_alternativa`, `reasignar_repartidor`, `avisar_cliente`, `compensar_reprogramar` |
| **Tabla Q** | 3 × 6 = 18 casillas. Arranca con valores predeterminados (conocimiento previo, p. ej. en `Lenta` vale más avisar al cliente que el despacho estándar) |
| **Recompensa / castigo** | Estrellas: r = (★ − 3) / 2 → 1★ = −1, 3★ = 0, 5★ = +1. Me gusta = +1, No me gusta = −1 |
| **Actualización** | Q(s,a) ← Q(s,a) + α · [ r + γ · maxₐ′ Q(s′,a′) − Q(s,a) ], con α = 0.3, γ = 0.9. Cada pedido es un episodio de un paso (s′ terminal), así que el objetivo es r |
| **Política ε-greedy** | Con probabilidad ε explora (acción al azar); si no, explota (mayor Q). ε = 0.5 · 0.95ⁿ (mínimo 0.05): con cada feedback explora menos |

La tabla se guarda en `q_tabla.json`, así el aprendizaje sigue entre sesiones y lo comparten el notebook y la página web.

In [ ]:
agente = AgenteQ(f"{CARPETA}/q_tabla.json", alfa=0.3, gamma=0.9, epsilon_inicial=0.5,
                 epsilon_min=0.05, decaimiento=0.95, semilla=int(time.time()))
print(f"Feedbacks guardados: {len(agente.historial)} | ε actual = {agente.epsilon:.3f}")
display(pd.DataFrame(agente.Q).round(3))

### Paso 7.1: ronda de feedback — el usuario califica

En cada vuelta se toma un pedido **real** del dataset, la red predice, el agente escoge la acción (exploración o explotación), el LLM escribe la recomendación y **el usuario escribe su calificación de 1 a 5**. Se muestra cómo cambia Q.

Criterios para calificar: ¿es coherente con la predicción?, ¿las acciones son concretas y útiles para ese pedido?, ¿la estrategia tiene sentido (p. ej. no ofrecer cupón a un pedido rápido)?

In [ ]:
df = pd.read_csv(CSV_PATH)
pool = df[RAW_COLS].dropna().sample(800, random_state=7).reset_index(drop=True)
pred_pool = predictor.predecir(pool).reset_index(drop=True)
pred_pool["estado"] = pred_pool["resultado"].map(estado_de)
print(pred_pool["estado"].value_counts())

def tomar_pedido(ronda):
    # alternamos los estados para que el agente aprenda en los tres
    estado = ESTADOS[ronda % 3]
    i = np.random.choice(np.where(pred_pool["estado"] == estado)[0])
    return pool.iloc[i], pred_pool.iloc[i]

In [ ]:
RONDAS = 9   # recomendaciones a calificar

for ronda in range(RONDAS):
    fila, pred = tomar_pedido(ronda)
    estado = estado_de(pred["resultado"])
    accion, modo = agente.elegir_accion(estado)
    texto = generar_descripcion_llm(llm, fila, pred, accion)

    print("=" * 100)
    print(f"RONDA {ronda+1}/{RONDAS} | estado: {estado} | acción: {accion} ({modo}, ε={agente.epsilon:.3f})")
    print(f"Pedido: {fila['Category']}, tráfico {fila['Traffic'].strip()}, clima {fila['Weather']}, "
          f"{fila['Vehicle'].strip()}, {fila['Order_Time']}")
    print(f"Predicción: {pred['resultado']} | {pred['minutos_estimados']:.0f} min | P(lenta) {pred['prob_lenta']:.0%}")
    print("-" * 100); print(texto); print("-" * 100)

    nota = ""
    while nota not in {"1", "2", "3", "4", "5"}:
        nota = input("Califica la recomendación de 1 a 5: ").strip()
    comentario = input("Comentario (opcional, Enter para omitir): ").strip()
    q_antes = agente.Q[estado][accion]
    r, q_nuevo = agente.feedback_estrellas(estado, accion, int(nota), comentario)
    print(f"→ {'RECOMPENSA' if r > 0 else 'CASTIGO' if r < 0 else 'NEUTRO'} r = {r:+.2f} | "
          f"Q({estado}, {accion}): {q_antes:.3f} → {q_nuevo:.3f}")

### Paso 7.2: ¿qué aprendió el agente?

- **Tabla Q** como mapa de calor (verde = recompensa esperada, rojo = castigo). La mejor acción de cada estado es la que el agente explota.
- **Recompensa por feedback** y su promedio acumulado.
- **ε por feedback**: cómo pasa de exploración a explotación.

In [ ]:
def graficar_agente(ag, titulo=""):
    hist = pd.DataFrame(ag.historial)
    Q = pd.DataFrame(ag.Q)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    im = axes[0].imshow(Q.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    axes[0].set_xticks(range(Q.shape[1]), Q.columns); axes[0].set_yticks(range(Q.shape[0]), Q.index)
    for i in range(Q.shape[0]):
        for j in range(Q.shape[1]):
            axes[0].text(j, i, f"{Q.values[i, j]:.2f}\nn={ag.N[Q.columns[j]][Q.index[i]]}", ha="center", va="center", fontsize=9)
    axes[0].set_title(f"Tabla Q {titulo}"); fig.colorbar(im, ax=axes[0])
    if len(hist):
        x = np.arange(1, len(hist) + 1)
        axes[1].bar(x, hist["recompensa"], color=np.where(hist["recompensa"] >= 0, "#2e8b57", "#c0392b"))
        axes[1].plot(x, hist["recompensa"].expanding().mean(), "k-", lw=2.5, label="promedio acumulado")
        axes[1].axhline(0, color="gray", lw=0.8); axes[1].set_ylim(-1.1, 1.1); axes[1].legend()
        axes[2].plot(x, hist["epsilon"], "o-")
    axes[1].set_title("Recompensa por feedback"); axes[1].set_xlabel("feedback")
    axes[2].set_title("ε (exploración → explotación)"); axes[2].set_xlabel("feedback"); axes[2].set_ylim(0, 0.55)
    plt.tight_layout(); plt.show()

graficar_agente(agente, "(feedback real)")
for e in ESTADOS:
    print(f"Mejor acción aprendida para '{e}': {agente.ranking(e)[0]}  →  {ACCIONES[agente.ranking(e)[0]]}")

### Paso 7.3: verificación del algoritmo con un usuario simulado

Con pocas calificaciones reales no se alcanza a ver toda la convergencia, así que comprobamos el algoritmo con un **usuario simulado** (agente aparte, **no se guarda** ni se mezcla con el feedback real). El usuario simulado tiene preferencias ocultas con ruido: en pedidos **rápidos** prefiere el despacho estándar, en **intermedios** la ruta alternativa y en **lentos** la compensación o reprogramación. Comparamos el agente Q-learning contra escoger la acción **al azar**.

In [ ]:
preferencias = {   # estrellas promedio que daría el usuario simulado (ocultas para el agente)
    "Rápida":     {"despacho_estandar": 4.6, "priorizar_tienda": 3.6, "ruta_alternativa": 2.6,
                   "reasignar_repartidor": 2.2, "avisar_cliente": 3.2, "compensar_reprogramar": 1.6},
    "Intermedia": {"despacho_estandar": 2.8, "priorizar_tienda": 3.4, "ruta_alternativa": 4.5,
                   "reasignar_repartidor": 3.3, "avisar_cliente": 3.9, "compensar_reprogramar": 2.4},
    "Lenta":      {"despacho_estandar": 1.5, "priorizar_tienda": 2.8, "ruta_alternativa": 3.4,
                   "reasignar_repartidor": 3.5, "avisar_cliente": 3.9, "compensar_reprogramar": 4.7},
}
rng = np.random.default_rng(0)
def usuario_simulado(estado, accion):
    return int(np.clip(round(rng.normal(preferencias[estado][accion], 0.8)), 1, 5))

sim = AgenteQ(ruta=None, semilla=0)          # ruta=None: no se guarda
estrellas_rl, estrellas_azar = [], []
for t in range(300):
    estado = ESTADOS[t % 3]
    accion, _ = sim.elegir_accion(estado)
    n = usuario_simulado(estado, accion); sim.feedback_estrellas(estado, accion, n); estrellas_rl.append(n)
    estrellas_azar.append(usuario_simulado(estado, rng.choice(list(ACCIONES))))

plt.figure(figsize=(10, 4))
plt.plot(pd.Series(estrellas_rl).rolling(30, min_periods=1).mean(), label="agente Q-learning")
plt.plot(pd.Series(estrellas_azar).rolling(30, min_periods=1).mean(), label="acción al azar")
plt.title("Usuario simulado: estrellas promedio (ventana de 30 feedbacks)"); plt.xlabel("feedback"); plt.ylabel("★")
plt.ylim(1, 5); plt.legend(); plt.show()

print(f"Promedio de los últimos 100 feedbacks → Q-learning: {np.mean(estrellas_rl[-100:]):.2f} ★ | azar: {np.mean(estrellas_azar[-100:]):.2f} ★")
for e in ESTADOS:
    real = max(preferencias[e], key=preferencias[e].get)
    print(f"{e:<11} aprendió: {sim.ranking(e)[0]:<22} | preferida real: {real:<22} | {'✔' if sim.ranking(e)[0] == real else '✘'}")
graficar_agente(sim, "(usuario simulado, 300 feedbacks)")

## Paso 8: recomendación con la política aprendida (explotación)

Para los tres pedidos del Paso 3, ahora la estrategia es la de **mayor Q** según el feedback real del usuario.

In [ ]:
for i in range(len(pedidos_demo)):
    fila, pred = pedidos_demo.iloc[i], pred_demo.iloc[i]
    estado = estado_de(pred["resultado"])
    accion = agente.ranking(estado)[0]
    texto = generar_descripcion_llm(llm, fila, pred, accion)
    print("=" * 100)
    print(f"PEDIDO {i+1}: {pred['resultado']} | {pred['minutos_estimados']:.0f} min | acción aprendida: {accion} "
          f"(Q={agente.Q[estado][accion]:.3f})")
    print("-" * 100); print(texto)

## Paso 9: despliegue web — API FastAPI + `index.html`

- `api.py`: servicio web con los puntos de acceso `GET /` (página), `GET /salud`, `GET /opciones`, `POST /predict` (red → agente RL → LLM) y `POST /feedback` (calificación → actualización de la tabla Q), `GET /q_tabla`.
- `static/index.html`: formulario del pedido, predicción, recomendación del LLM, calificación con estrellas, 👍/👎 para las otras estrategias del agente y la tabla Q en vivo.

En Colab la API corre en segundo plano y se publica con un túnel de **Cloudflare** (link `https://….trycloudflare.com`, sin crear cuenta). Usa **el mismo agente** del notebook, así que el feedback de la web también entra en las gráficas del Paso 10.

In [ ]:
%%writefile despliegue_llm_rl/api.py
"""API web: formulario -> red neuronal -> agente RL -> LLM -> recomendación -> feedback -> tabla Q.

Correr local:   uvicorn api:app --host 0.0.0.0 --port 8000     ->  http://127.0.0.1:8000
"""
import os
import time

import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

from features import RAW_COLS
from recomendador import MODELO_LLM, cargar_llm, estado_de, generar_descripcion_llm
from rl_feedback import ACCIONES, AgenteQ

CARPETA = os.path.dirname(os.path.abspath(__file__))


class Pedido(BaseModel):
    Agent_Age: float
    Agent_Rating: float
    Store_Latitude: float
    Store_Longitude: float
    Drop_Latitude: float
    Drop_Longitude: float
    Order_Date: str
    Order_Time: str
    Pickup_Time: str
    Weather: str
    Traffic: str
    Vehicle: str
    Area: str
    Category: str


class Feedback(BaseModel):
    estado: str
    accion: str
    estrellas: int | None = None      # calificación de la recomendación del LLM (1 a 5)
    me_gusta: bool | None = None      # me gusta / no me gusta de una acción del ranking
    comentario: str = ""


def crear_app(predictor=None, llm=None, agente=None):
    if predictor is None:
        from inferencia import Predictor
        predictor = Predictor(CARPETA)
    if agente is None:
        agente = AgenteQ(os.path.join(CARPETA, "q_tabla.json"))
    if llm is None:
        llm = cargar_llm()

    app = FastAPI(title="Proyecto 8: red neuronal + LLM + aprendizaje por refuerzo")
    # Permite que la página publicada en Vercel (otro dominio) llame a esta API en Colab.
    app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

    @app.get("/")
    def inicio():
        return FileResponse(os.path.join(CARPETA, "static", "index.html"))

    @app.get("/salud")
    def salud():
        return {"ok": True, "modelo_neuronal": predictor.cfg["modelo"], "llm": MODELO_LLM,
                "epsilon": round(agente.epsilon, 3), "feedbacks": len(agente.historial)}

    @app.get("/opciones")
    def opciones():
        return {"opciones": predictor.cfg["opciones"], "acciones": ACCIONES}

    @app.post("/predict")
    def predict(pedido: Pedido):
        fila = pd.Series(pedido.model_dump())[RAW_COLS]
        pred = predictor.predecir(pd.DataFrame([fila])).iloc[0]
        estado = estado_de(pred["resultado"])
        if estado is None or pd.isna(pred["minutos_estimados"]):
            raise HTTPException(422, f"{pred['resultado']}. {pred.get('motivo', '')}".strip())

        accion, modo = agente.elegir_accion(estado)
        t0 = time.time()
        texto = generar_descripcion_llm(llm, fila, pred, accion)
        ranking = [{"accion": a, "texto": ACCIONES[a], "Q": round(agente.Q[estado][a], 3)}
                   for a in agente.ranking(estado) if a != accion][:3]   # sin repetir la ya escogida
        return {
            "prediccion": {"resultado": pred["resultado"], "minutos_estimados": round(float(pred["minutos_estimados"]), 1),
                           "prob_lenta": round(float(pred["prob_lenta"]), 4)},
            "estado": estado, "accion": accion, "accion_texto": ACCIONES[accion], "modo": modo,
            "epsilon": round(agente.epsilon, 3),
            "recomendacion_llm": texto, "segundos_llm": round(time.time() - t0, 1),
            "ranking_rl": ranking,
        }

    @app.post("/feedback")
    def feedback(fb: Feedback):
        if fb.estado not in agente.Q or fb.accion not in ACCIONES:
            raise HTTPException(400, "estado o acción desconocidos")
        if fb.estrellas is not None:
            r, q = agente.feedback_estrellas(fb.estado, fb.accion, fb.estrellas, fb.comentario)
        elif fb.me_gusta is not None:
            r, q = agente.feedback_me_gusta(fb.estado, fb.accion, fb.me_gusta)
        else:
            raise HTTPException(400, "envía 'estrellas' o 'me_gusta'")
        return {"recompensa": r, "q_nuevo": round(q, 4), "epsilon": round(agente.epsilon, 3),
                "feedbacks": len(agente.historial), "tabla_q": agente.tabla()}

    @app.get("/q_tabla")
    def q_tabla():
        return {"epsilon": round(agente.epsilon, 3), "feedbacks": len(agente.historial),
                "tabla_q": agente.tabla()}

    app.state.agente, app.state.predictor, app.state.llm = agente, predictor, llm
    return app


# En el notebook se pone API_AUTOLOAD=0 para crear la app con los objetos ya cargados.
if os.environ.get("API_AUTOLOAD", "1") == "1":
    app = crear_app()

In [ ]:
%%writefile despliegue_llm_rl/static/index.html
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Entrega Amazon: red + LLM + RL</title>
<style>
  :root { --fondo:#f4f6f8; --tarjeta:#fff; --texto:#1d2733; --suave:#5b6876; --borde:#dfe4ea;
          --primario:#232f3e; --acento:#ff9900; --bien:#1f8a4c; --mal:#c0392b; --medio:#b7791f; }
  * { box-sizing:border-box; }
  body { margin:0; font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif; background:var(--fondo); color:var(--texto); }
  header { background:var(--primario); color:#fff; padding:18px 16px; }
  header h1 { margin:0; font-size:20px; }
  header p { margin:4px 0 0; font-size:13px; opacity:.85; }
  main { max-width:1100px; margin:0 auto; padding:16px; display:grid; gap:16px; grid-template-columns:minmax(0,1fr) minmax(0,1.25fr); }
  @media (max-width:860px){ main { grid-template-columns:1fr; } }
  .tarjeta { background:var(--tarjeta); border:1px solid var(--borde); border-radius:12px; padding:16px; }
  h2 { font-size:16px; margin:0 0 12px; }
  form { display:grid; grid-template-columns:1fr 1fr; gap:10px; }
  label { display:flex; flex-direction:column; gap:4px; font-size:13px; color:var(--suave); }
  input, select, textarea { padding:7px 8px; font-size:14px; border:1px solid var(--borde); border-radius:6px; color:var(--texto); background:#fff; width:100%; }
  .fila-botones { grid-column:1/-1; display:flex; gap:8px; flex-wrap:wrap; }
  button { cursor:pointer; border:0; border-radius:8px; padding:9px 14px; font-size:14px; }
  .principal { background:var(--primario); color:#fff; flex:1; }
  .ejemplo { background:#e9edf2; color:var(--texto); }
  .principal:disabled { opacity:.6; cursor:wait; }
  .Rápida{--c:var(--bien); --c-suave:#e7f5ec;} .Intermedia{--c:var(--medio); --c-suave:#fbf1e1;} .Lenta{--c:var(--mal); --c-suave:#fbe9e7;}
  .dato { font-size:14px; color:var(--suave); }
  .dato b { color:var(--texto); }
  .seccion { font-size:12px; font-weight:700; letter-spacing:.06em; text-transform:uppercase; color:var(--suave); margin:20px 0 8px; }

  /* --- Predicción --- */
  .hero { border-radius:12px; padding:16px; background:linear-gradient(135deg, var(--c-suave), #fff 75%); border:1px solid var(--borde); border-left:6px solid var(--c); }
  .hero-fila { display:flex; justify-content:space-between; align-items:flex-start; gap:12px; flex-wrap:wrap; }
  .chip { display:inline-block; padding:5px 12px; border-radius:999px; font-weight:700; font-size:13px; color:#fff; background:var(--c); }
  .minutos { font-size:44px; font-weight:800; line-height:1; color:var(--c); margin-top:10px; font-variant-numeric:tabular-nums; }
  .minutos small { font-size:15px; font-weight:600; color:var(--suave); margin-left:4px; }
  .prob { min-width:170px; flex:0 1 220px; }
  .prob .etq { display:flex; justify-content:space-between; font-size:12.5px; color:var(--suave); margin-bottom:5px; }
  .prob .etq b { color:var(--texto); }
  .barra { height:8px; border-radius:99px; background:#e6eaef; overflow:hidden; }
  .barra > i { display:block; height:100%; border-radius:99px; background:var(--c); transition:width .6s ease; }
  .linea { position:relative; margin-top:18px; }
  .zonas { display:flex; height:10px; border-radius:99px; overflow:hidden; }
  .zonas span:nth-child(1){ flex:70; background:#bfe3cc; } .zonas span:nth-child(2){ flex:110; background:#f2d9ab; } .zonas span:nth-child(3){ flex:60; background:#f1c1bb; }
  .marca { position:absolute; top:-5px; width:20px; height:20px; margin-left:-10px; border-radius:50%; background:#fff; border:4px solid var(--c); box-shadow:0 1px 4px rgba(0,0,0,.25); transition:left .6s ease; }
  .escala { position:relative; height:14px; font-size:11px; color:var(--suave); margin-top:6px; }
  .escala span { position:absolute; transform:translateX(-50%); white-space:nowrap; }
  .escala span:first-child { transform:none; } .escala span:last-child { transform:translateX(-100%); }

  /* --- Recomendación del LLM --- */
  .diag { display:flex; gap:10px; background:#f3f6fa; border-radius:10px; padding:12px 14px; line-height:1.5; font-size:14.5px; }
  .diag .ico { font-size:20px; line-height:1.2; }
  .pasos { list-style:none; margin:0; padding:0; display:grid; gap:8px; counter-reset:paso; }
  .pasos li { counter-increment:paso; display:flex; gap:10px; align-items:flex-start; padding:10px 12px; border:1px solid var(--borde); border-radius:10px; font-size:14px; line-height:1.45; background:#fff; }
  .pasos li::before { content:counter(paso); flex:0 0 24px; height:24px; border-radius:50%; display:grid; place-items:center; font-size:12px; font-weight:700; color:#fff; background:var(--primario); }
  .pasos li.principal-acc { border-color:var(--acento); background:#fff8ec; }
  .pasos li.principal-acc::before { background:var(--acento); }
  .burbuja { position:relative; background:var(--primario); color:#fff; border-radius:14px 14px 14px 4px; padding:12px 14px; font-size:14px; line-height:1.5; max-width:95%; }
  .burbuja .de { display:block; font-size:11px; opacity:.7; margin-bottom:4px; text-transform:uppercase; letter-spacing:.05em; }
  .llm { line-height:1.55; font-size:14.5px; background:#fbfcfd; border:1px solid var(--borde); border-radius:10px; padding:12px; }

  /* --- Agente RL --- */
  .rl { display:flex; flex-wrap:wrap; gap:6px; margin-top:14px; }
  .pill { font-size:12px; padding:4px 10px; border-radius:999px; background:#eef1f5; color:var(--texto); }
  .pill.estr { background:#232f3e; color:#fff; }
  .pill.explo { background:#e8f0fe; color:#1a56c4; }
  .pill.explota { background:#e7f5ec; color:var(--bien); }

  /* --- Feedback --- */
  .caja-fb { border:1px solid var(--borde); border-radius:12px; padding:14px; background:#fcfdfe; }
  .estrellas { display:flex; gap:6px; font-size:32px; margin:0 0 8px; }
  .estrellas span { cursor:pointer; color:#d3d9e0; user-select:none; transition:transform .12s, color .12s; }
  .estrellas span:hover { transform:scale(1.15); }
  .estrellas span.on { color:var(--acento); }
  .acciones-rl { list-style:none; padding:0; margin:0; display:grid; gap:8px; }
  .acciones-rl li { display:grid; grid-template-columns:1fr auto; gap:6px 10px; align-items:center; border:1px solid var(--borde); border-radius:10px; padding:10px 12px; font-size:13.5px; }
  .acciones-rl .q { grid-column:1; display:flex; align-items:center; gap:8px; font-size:12px; color:var(--suave); }
  .acciones-rl .q .barra { flex:1; max-width:160px; height:6px; }
  .acciones-rl .q .barra > i { background:var(--acento); }
  .acciones-rl .votos { grid-row:1 / span 2; grid-column:2; display:flex; gap:6px; }
  .acciones-rl button { padding:6px 10px; background:#eef1f5; font-size:15px; }
  .acciones-rl button:hover { background:#e1e6ec; }

  /* --- Cargando --- */
  .cargando { display:grid; gap:10px; }
  .cargando i { display:block; height:14px; border-radius:6px; background:linear-gradient(90deg,#eef1f5 25%,#e2e7ed 50%,#eef1f5 75%); background-size:200% 100%; animation:brillo 1.2s infinite; }
  .cargando i.alto { height:90px; border-radius:12px; }
  @keyframes brillo { to { background-position:-200% 0; } }
  @media (prefers-reduced-motion:reduce){ .cargando i, .barra > i, .marca { animation:none; transition:none; } }
  table { border-collapse:collapse; width:100%; font-size:12.5px; }
  th, td { border:1px solid var(--borde); padding:5px 6px; text-align:center; }
  th:first-child, td:first-child { text-align:left; }
  .tabla-scroll { overflow-x:auto; }
  .aviso { font-size:13px; margin-top:8px; }
  .servidor { margin-top:8px; font-size:13px; color:var(--suave); }
  .servidor summary { cursor:pointer; }
  .fila-api { display:flex; gap:8px; }
  .oculto { display:none; }
  .error { color:var(--mal); }
</style>
</head>
<body>
<header>
  <h1>¿Cómo llegará el pedido? Red neuronal + LLM + Aprendizaje por refuerzo</h1>
  <p>Proyecto 8 · Kevin Tapias, Sebastian Villera y Anuar Marriaga · Rápida ≤ 70 min · Lenta ≥ 180 min</p>
</header>
<main>
  <section class="tarjeta">
    <h2>1. Datos del pedido</h2>
    <form id="form">
      <label>Edad del repartidor<input name="Agent_Age" type="number" value="30" required></label>
      <label>Calificación (1-5)<input name="Agent_Rating" type="number" step="0.1" value="4.6" required></label>
      <label>Latitud tienda<input name="Store_Latitude" type="number" step="any" value="12.914" required></label>
      <label>Longitud tienda<input name="Store_Longitude" type="number" step="any" value="77.678" required></label>
      <label>Latitud destino<input name="Drop_Latitude" type="number" step="any" value="12.954" required></label>
      <label>Longitud destino<input name="Drop_Longitude" type="number" step="any" value="77.718" required></label>
      <label>Fecha<input name="Order_Date" type="date" value="2022-03-19" required></label>
      <label>Hora del pedido<input name="Order_Time" type="time" value="18:30" required></label>
      <label>Hora de recogida<input name="Pickup_Time" type="time" value="18:45" required></label>
      <label>Clima<select name="Weather"></select></label>
      <label>Tráfico<select name="Traffic"></select></label>
      <label>Vehículo<select name="Vehicle"></select></label>
      <label>Área<select name="Area"></select></label>
      <label>Categoría<select name="Category"></select></label>
      <div class="fila-botones">
        <button type="button" class="ejemplo" data-ej="rapido">Ejemplo rápido</button>
        <button type="button" class="ejemplo" data-ej="intermedio">Ejemplo intermedio</button>
        <button type="button" class="ejemplo" data-ej="lento">Ejemplo lento</button>
      </div>
      <div class="fila-botones"><button class="principal" id="btn">Predecir y recomendar</button></div>
    </form>
    <p id="estadoApi" class="aviso"></p>
    <details id="cajaApi" class="servidor">
      <summary>Servidor de la API</summary>
      <p class="aviso">Si abres esta página desde Vercel, pega aquí el link <b>https://….trycloudflare.com</b> que imprime el Paso 9 del notebook en Colab.</p>
      <div class="fila-api"><input id="urlApi" type="url" placeholder="https://….trycloudflare.com"><button type="button" class="ejemplo" id="conectar">Conectar</button></div>
    </details>
  </section>

  <section class="tarjeta">
    <h2>2. Predicción y recomendación</h2>
    <div id="vacio" class="dato">Llena el formulario y pulsa <b>Predecir y recomendar</b>. El LLM tarda unos segundos.</div>
    <div id="cargando" class="cargando oculto"><i class="alto"></i><i style="width:90%"></i><i style="width:75%"></i><i style="width:82%"></i><i style="width:60%"></i></div>
    <div id="resultado" class="oculto">
      <div id="hero" class="hero">
        <div class="hero-fila">
          <div>
            <span id="chip" class="chip"></span>
            <div class="minutos"><span id="minutos"></span><small>min estimados</small></div>
          </div>
          <div class="prob">
            <div class="etq"><span>Probabilidad de entrega lenta</span><b id="prob"></b></div>
            <div class="barra"><i id="probBarra"></i></div>
          </div>
        </div>
        <div class="linea" aria-hidden="true">
          <div class="zonas"><span></span><span></span><span></span></div>
          <div class="marca" id="marca"></div>
        </div>
        <div class="escala"><span style="left:0">0 min</span><span style="left:29.17%">70</span><span style="left:75%">180</span><span style="left:100%">240+</span></div>
      </div>

      <div id="llm"></div>
      <div class="rl">
        <span class="pill estr" id="accion" title="Estrategia escogida por el agente de Q-learning"></span>
        <span class="pill" id="modo"></span>
        <span class="pill">ε = <span id="eps"></span></span>
        <span class="pill">LLM: <span id="seg"></span> s</span>
      </div>

      <h2 style="margin-top:20px">3. Califica la recomendación (feedback)</h2>
      <div class="caja-fb">
        <div class="estrellas" id="estrellas"><span data-v="1">★</span><span data-v="2">★</span><span data-v="3">★</span><span data-v="4">★</span><span data-v="5">★</span></div>
        <textarea id="comentario" rows="2" placeholder="Comentario opcional"></textarea>
        <div class="fila-botones" style="margin-top:8px"><button class="principal" id="enviar">Enviar calificación</button></div>
        <p id="msgFb" class="aviso"></p>
      </div>

      <div class="seccion">Otras estrategias del agente (mayor Q)</div>
      <ul class="acciones-rl" id="ranking"></ul>
    </div>
  </section>

  <section class="tarjeta" style="grid-column:1/-1">
    <h2>Tabla Q del agente (lo que ha aprendido) · <span id="resumenQ" class="dato"></span></h2>
    <div class="tabla-scroll"><table id="tablaQ"></table></div>
  </section>
</main>

<script>
const EJEMPLOS = {
  rapido:     {Agent_Age:25, Agent_Rating:4.9, Store_Latitude:12.914, Store_Longitude:77.678, Drop_Latitude:12.934, Drop_Longitude:77.688, Order_Date:"2022-03-15", Order_Time:"10:00", Pickup_Time:"10:10", Weather:"Sunny",  Traffic:"Low", Vehicle:"motorcycle", Area:"Urban", Category:"Grocery"},
  intermedio: {Agent_Age:30, Agent_Rating:4.6, Store_Latitude:12.914, Store_Longitude:77.678, Drop_Latitude:12.954, Drop_Longitude:77.718, Order_Date:"2022-03-19", Order_Time:"18:30", Pickup_Time:"18:45", Weather:"Fog",    Traffic:"Jam", Vehicle:"motorcycle", Area:"Metropolitian", Category:"Clothing"},
  lento:      {Agent_Age:38, Agent_Rating:4.1, Store_Latitude:22.745, Store_Longitude:75.892, Drop_Latitude:22.875, Drop_Longitude:76.022, Order_Date:"2022-03-20", Order_Time:"19:30", Pickup_Time:"19:45", Weather:"Stormy", Traffic:"Jam", Vehicle:"scooter", Area:"Metropolitian", Category:"Electronics"},
};
const $ = (id) => document.getElementById(id);
let API = "";
try { API = (new URLSearchParams(location.search).get("api") || localStorage.getItem("api") || "").replace(/\/+$/, ""); } catch {}
if (location.hostname.endsWith("trycloudflare.com") || location.hostname === "localhost" || location.hostname === "127.0.0.1") API = "";
const url = (ruta) => (API ? API + "/" : "") + ruta;
const form = $("form");
let ultimo = null, estrellas = 0, ACC = {};

function escapar(s){ return s.replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c])); }
function md(s){ return escapar(s).replace(/\*\*(.+?)\*\*/g, "<b>$1</b>").replace(/\n/g, "<br>"); }

function llenarForm(d){ for (const [k,v] of Object.entries(d)) if (form.elements[k]) form.elements[k].value = v; }

async function iniciar(){
  try {
    const r = await (await fetch(url("opciones"))).json();
    ACC = r.acciones;
    for (const [campo, valores] of Object.entries(r.opciones)) {
      form.elements[campo].innerHTML = valores.map(v => `<option>${escapar(v)}</option>`).join("");
    }
    llenarForm(EJEMPLOS.intermedio);
    const s = await (await fetch(url("salud"))).json();
    $("estadoApi").textContent = `API lista · red: ${s.modelo_neuronal} · LLM: ${s.llm}`;
    pintarTabla(await (await fetch(url("q_tabla"))).json());
    $("cajaApi").open = false;
  } catch(e) {
    $("estadoApi").innerHTML = `<span class="error">No se pudo conectar con la API${API ? " (" + escapar(API) + ")" : ""}. Revisa que el notebook esté corriendo en Colab y pega su link abajo.</span>`;
    $("cajaApi").open = true;
  }
}

// Servidor de la API: vacío = el mismo sitio (cuando la página la sirve FastAPI en Colab);
// en Vercel se usa el link del túnel de Colab (?api=… en la URL o el campo de abajo).
function guardarApi(v){ API = (v || "").trim().replace(/\/+$/, ""); try { localStorage.setItem("api", API); } catch {} }
$("urlApi").value = API;
$("conectar").onclick = () => { guardarApi($("urlApi").value); $("estadoApi").textContent = "Conectando…"; iniciar(); };

document.querySelectorAll("[data-ej]").forEach(b => b.onclick = () => llenarForm(EJEMPLOS[b.dataset.ej]));

form.onsubmit = async (ev) => {
  ev.preventDefault();
  const datos = Object.fromEntries(new FormData(form));
  for (const k of ["Agent_Age","Agent_Rating","Store_Latitude","Store_Longitude","Drop_Latitude","Drop_Longitude"]) datos[k] = parseFloat(datos[k]);
  $("btn").disabled = true; $("btn").textContent = "Pensando… (red + LLM)";
  $("vacio").classList.add("oculto"); $("resultado").classList.add("oculto"); $("cargando").classList.remove("oculto");
  try {
    const resp = await fetch(url("predict"), {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(datos)});
    const r = await resp.json();
    if (!resp.ok) throw new Error(r.detail || resp.statusText);
    ultimo = r; estrellas = 0; pintarEstrellas(); $("msgFb").textContent = ""; $("comentario").value = "";
    $("resultado").classList.remove("oculto");
    const min = r.prediccion.minutos_estimados, p = r.prediccion.prob_lenta;
    $("hero").className = "hero " + r.estado;
    $("chip").textContent = r.prediccion.resultado;
    $("minutos").textContent = Math.round(min);
    $("prob").textContent = (p*100).toFixed(1) + "%";
    $("probBarra").style.width = (p*100).toFixed(1) + "%";
    $("marca").style.left = Math.min(Math.max(min, 0), 240) / 240 * 100 + "%";
    $("llm").innerHTML = pintarRecomendacion(r.recomendacion_llm, r.estado);
    $("accion").textContent = "Estrategia: " + (ACC[r.accion] ? r.accion.replace(/_/g, " ") : r.accion);
    $("accion").title = ACC[r.accion] || "";
    $("modo").textContent = r.modo; $("modo").className = "pill " + (r.modo === "exploración" ? "explo" : "explota");
    $("eps").textContent = r.epsilon; $("seg").textContent = r.segundos_llm;
    const qMax = Math.max(1e-9, ...r.ranking_rl.map(a => Math.abs(a.Q)));
    $("ranking").innerHTML = r.ranking_rl.map(a => `<li><span>${escapar(a.texto)}</span>
      <span class="votos"><button data-a="${a.accion}" data-g="1" title="Me gusta">👍</button><button data-a="${a.accion}" data-g="0" title="No me gusta">👎</button></span>
      <span class="q">Q = ${a.Q}<span class="barra"><i style="width:${Math.max(a.Q, 0) / qMax * 100}%"></i></span></span></li>`).join("");
    $("ranking").querySelectorAll("button").forEach(b => b.onclick = () => enviarFb({accion:b.dataset.a, me_gusta:b.dataset.g==="1"}, b));
  } catch(e) { $("vacio").classList.remove("oculto"); $("vacio").innerHTML = `<span class="error">${escapar(String(e.message||e))}</span>`; }
  $("cargando").classList.add("oculto");
  $("btn").disabled = false; $("btn").textContent = "Predecir y recomendar";
};

// Separa la respuesta del LLM en diagnóstico (1.), acciones (2.) y mensaje al cliente (3.).
// Si el texto no sigue ese formato, se muestra tal cual.
function pintarRecomendacion(texto, estado){
  const partes = {};
  let actual = null;
  const SECCION = {d:"1", a:"2", m:"3"};
  for (const linea of texto.split("\n")) {
    // "1. **Diagnóstico:** …", "- **Acciones:**", "**Mensaje para el cliente:** …" o solo "2. …"
    const e = linea.match(/^\s*(?:[-*•]|\d[.)])?\s*\**\s*(diagn[oó]stico|acciones|mensaje)[^:\n]{0,30}:\s*\**\s*(.*)$/i);
    const n = linea.match(/^\s*(?:\*\*)?\s*([123])[.)]\s*(?:\*\*)?\s*(.*)$/);
    const clave = e ? SECCION[e[1][0].toLowerCase()] : (n ? n[1] : null);
    if (clave && !partes[clave]) { actual = clave; partes[actual] = [e ? e[2] : n[2]]; }
    else if (actual) partes[actual].push(linea);
  }
  if (!partes["1"] || !partes["2"]) return `<div class="llm">${md(texto)}</div>`;
  const sinEtiqueta = s => s.replace(/^\s*\**\s*(diagn[oó]stico|acciones|mensaje[^:]*)\s*:?\s*\**\s*:?\s*/i, "").trim();
  const diag = sinEtiqueta(partes["1"].join(" ").replace(/\s+/g, " "));
  const acciones = partes["2"].map(l => l.trim()).filter(l => /^[-*•]/.test(l))
    .map(l => l.replace(/^[-*•]\s*/, "")).filter(Boolean);
  const mensaje = partes["3"] ? sinEtiqueta(partes["3"].join(" ").replace(/\s+/g, " ")).replace(/^["“]|["”]$/g, "") : "";
  if (!acciones.length) return `<div class="llm">${md(texto)}</div>`;
  const ico = {"Rápida":"✅", "Intermedia":"⏱️", "Lenta":"⚠️"}[estado] || "📦";
  return `<div class="seccion">Diagnóstico del LLM</div>
    <div class="diag"><span class="ico">${ico}</span><div>${md(diag)}</div></div>
    <div class="seccion">Acciones recomendadas</div>
    <ol class="pasos">${acciones.map((a, i) => `<li class="${i === 0 ? "principal-acc" : ""}"><div>${md(a)}</div></li>`).join("")}</ol>
    ${mensaje ? `<div class="seccion">Mensaje para el cliente</div><div class="burbuja"><span class="de">Servicio al cliente</span>${md(mensaje)}</div>` : ""}`;
}

function pintarEstrellas(){ $("estrellas").querySelectorAll("span").forEach(s => s.classList.toggle("on", +s.dataset.v <= estrellas)); }
$("estrellas").querySelectorAll("span").forEach(s => s.onclick = () => { estrellas = +s.dataset.v; pintarEstrellas(); });

async function enviarFb(extra, boton){
  if (!ultimo) return;
  const cuerpo = {estado: ultimo.estado, accion: ultimo.accion, comentario: $("comentario").value, ...extra};
  const resp = await fetch(url("feedback"), {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(cuerpo)});
  const r = await resp.json();
  if (!resp.ok) { $("msgFb").innerHTML = `<span class="error">${escapar(r.detail)}</span>`; return; }
  const signo = r.recompensa > 0 ? "recompensa" : (r.recompensa < 0 ? "castigo" : "neutro");
  $("msgFb").textContent = `✔ ${signo} r = ${r.recompensa} → Q(${ultimo.estado}, ${cuerpo.accion}) = ${r.q_nuevo} · ε = ${r.epsilon} · feedbacks: ${r.feedbacks}`;
  if (boton) boton.parentElement.innerHTML = r.recompensa > 0 ? "👍 gracias" : "👎 anotado";
  pintarTabla(r);
}
$("enviar").onclick = () => {
  if (!estrellas) { $("msgFb").textContent = "Escoge de 1 a 5 estrellas."; return; }
  enviarFb({estrellas});
};

function pintarTabla(r){
  const estados = ["Rápida","Intermedia","Lenta"];
  const filas = {}; r.tabla_q.forEach(f => (filas[f.accion] ??= {})[f.estado] = f);
  const color = q => q >= 0 ? `rgba(31,138,76,${Math.min(Math.abs(q),1)*0.45})` : `rgba(192,57,43,${Math.min(Math.abs(q),1)*0.45})`;
  const mejores = Object.fromEntries(estados.map(e => [e, Object.keys(filas).reduce((a,b) => filas[a][e].Q >= filas[b][e].Q ? a : b)]));
  $("tablaQ").innerHTML = `<tr><th>Acción</th>${estados.map(e => `<th>${e}</th>`).join("")}</tr>` +
    Object.entries(filas).map(([a, fs]) => `<tr><td>${escapar(ACC[a] || a)}</td>${estados.map(e =>
      `<td style="background:${color(fs[e].Q)}">${mejores[e]===a ? "<b>" : ""}${fs[e].Q.toFixed(3)}${mejores[e]===a ? "</b>" : ""}<br><span class="dato">n=${fs[e].veces}</span></td>`).join("")}</tr>`).join("");
  $("resumenQ").textContent = `ε = ${r.epsilon} · ${r.feedbacks} feedbacks · en negrita la mejor acción por estado`;
}
iniciar();
</script>
</body>
</html>

In [ ]:
import threading, uvicorn
os.environ["API_AUTOLOAD"] = "0"          # la app se crea con los objetos ya cargados
import api; importlib.reload(api)
app = api.crear_app(predictor=predictor, llm=llm, agente=agente)

if "servidor" in globals():               # si se vuelve a correr la celda, se apaga el servidor anterior
    servidor.should_exit = True; time.sleep(3)
servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(3)

import requests
print(requests.get("http://127.0.0.1:8000/salud").json())

In [ ]:
# Prueba de la API como lo haría la página: POST /predict y POST /feedback
pedido = pedidos_demo.iloc[2].to_dict()
r = requests.post("http://127.0.0.1:8000/predict", json=pedido).json()
print(json.dumps({k: v for k, v in r.items() if k != "recomendacion_llm"}, ensure_ascii=False, indent=1))
print("\nRECOMENDACIÓN DEL LLM:\n" + r["recomendacion_llm"])

In [ ]:
# Link público (túnel de Cloudflare). Abrirlo, llenar el formulario y calificar.
import subprocess, re
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("cloudflared", 0o755)
tunel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for _ in range(200):
    linea = tunel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linea)
    if m:
        URL = m.group(0); break
print("Página web pública:", URL)

# Alternativa sin túnel (solo funciona en este navegador):
from google.colab.output import eval_js
print("Link de Colab:", eval_js("google.colab.kernel.proxyPort(8000)"))

## Paso 10: resultados finales del aprendizaje por refuerzo y descarga

Volvemos a graficar con **todo** el feedback (notebook + página web) y descargamos la app completa (`despliegue_llm_rl.zip`, con la tabla Q aprendida) y el registro de calificaciones.

In [ ]:
hist = pd.DataFrame(agente.historial)
hist.to_csv(f"{CARPETA}/feedback_usuarios.csv", index=False)
print(f"Feedbacks totales: {len(hist)}")
if len(hist):
    display(hist[["fecha", "estado", "accion", "tipo", "recompensa", "q_antes", "q_despues", "epsilon"]].tail(15).round(3))
    mitad = len(hist) // 2
    if mitad:
        print(f"Recompensa promedio — primera mitad: {hist['recompensa'][:mitad].mean():+.3f} | "
              f"segunda mitad: {hist['recompensa'][mitad:].mean():+.3f}")
graficar_agente(agente, "(final)")

In [ ]:
shutil.make_archive("despliegue_llm_rl", "zip", CARPETA)
files.download("despliegue_llm_rl.zip")
files.download(f"{CARPETA}/feedback_usuarios.csv")

## Conclusiones

- **Modelo neuronal → LLM por pipeline:** la predicción de la red (categoría, minutos y probabilidad de entrega lenta) entra al prompt junto con los datos del pedido y la estrategia del agente; el LLM pre-entrenado **Qwen2.5-3B-Instruct** la convierte en recomendaciones concretas para la tienda, el repartidor y el cliente. El rol del sistema y la verificación de coherencia hacen que el texto respete la predicción y no invente datos.
- **Aprendizaje por refuerzo con feedback humano:** la calificación del usuario es la recompensa (o castigo) del **Q-learning**; la tabla Q de 3 estados × 6 acciones se actualiza con α y γ, y la política ε-greedy pasa de explorar a explotar. Con el usuario simulado el agente encontró la acción preferida en los tres estados y superó claramente a escoger al azar.
- **Despliegue:** API FastAPI (`/predict`, `/feedback`) + `index.html`; el mismo código corre local (`uvicorn api:app`) o en Colab con link público.
- **Límite:** el LLM no se re-entrena; lo que aprende el agente es **qué estrategia pedirle** al LLM para cada tipo de predicción, que es la forma práctica de aplicar RL con feedback sin GPU de entrenamiento.